<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/text/tutorials/word2vec"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/text/blob/master/docs/tutorials/word2vec.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/text/blob/master/docs/tutorials/word2vec.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/text/docs/tutorials/word2vec.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
</table>

# word2vec

In [7]:
import io
import re
import string
import tqdm

import numpy as np

import tensorflow as tf
from tensorflow.keras import layers

%load_ext tensorboard
AUTOTUNE = tf.data.AUTOTUNE

SEED = 42
DIM_VECTORES = 128

path_to_file = "/content/cleaned_PHMRC_VAI_redacted_free_text.test_blind.csv"

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


## Explicación

### Vectorización de una frase

Ejemplo, la siguiente frase:

> The wide road shimmered in the hot sun.

Tokenizar la frase:

In [ ]:
sentence = "The wide road shimmered in the hot sun"
tokens = list(sentence.lower().split())
print(len(tokens))

8


Crear un vocabulario para asociar palabras -> index

In [ ]:
vocab, index = {}, 1  # start indexing from 1
vocab['<pad>'] = 0  # add a padding token
for token in tokens:
  if token not in vocab:
    vocab[token] = index
    index += 1
vocab_size = len(vocab)
print(vocab)

{'<pad>': 0, 'the': 1, 'wide': 2, 'road': 3, 'shimmered': 4, 'in': 5, 'hot': 6, 'sun': 7}


Crear el vocabulario inverso: index -> palabras

In [ ]:
inverse_vocab = {index: token for token, index in vocab.items()}
print(inverse_vocab)

{0: '<pad>', 1: 'the', 2: 'wide', 3: 'road', 4: 'shimmered', 5: 'in', 6: 'hot', 7: 'sun'}


Vectorizar la frase

In [ ]:
example_sequence = [vocab[word] for word in tokens]
print(example_sequence)

[1, 2, 3, 4, 5, 1, 6, 7]


### Generación de skip-grams positivos desde una frase

Para cada palabra, **guarda el skip-gram positivo** (par `[palabra_seleccionada,palabra_cercana]`) para asociarlas entre si.

Es importante elegir cuántos pares se quieren guardar, `window_size` es las palabras que se toman como "palabras cercanas".

Por ejemplo, window_size=2 significa que las 2 palabras de la izquierda y las 2 de la derecha se consideran cercanas.

In [ ]:
window_size = 2
positive_skip_grams, _ = tf.keras.preprocessing.sequence.skipgrams(
      example_sequence,
      vocabulary_size=vocab_size,
      window_size=window_size,
      negative_samples=0,
      seed=SEED)
print((positive_skip_grams))

[[5, 6], [4, 1], [4, 2], [7, 6], [1, 6], [3, 2], [3, 1], [4, 3], [5, 1], [6, 1], [4, 5], [1, 5], [1, 3], [5, 4], [6, 7], [5, 3], [2, 1], [1, 4], [7, 1], [2, 4], [6, 5], [3, 4], [3, 5], [1, 2], [2, 3], [1, 7]]


Imprimir varios skip-grams

In [ ]:
for target, context in positive_skip_grams[:5]:
  print(f"({target}, {context}): ({inverse_vocab[target]}, {inverse_vocab[context]})")

(5, 6): (in, hot)
(4, 1): (shimmered, the)
(4, 2): (shimmered, wide)
(7, 6): (sun, hot)
(1, 6): (the, hot)


### Generar skip-grams negativos a partir de uno positivo

A partir de un skip-gram positivo, **genera un skip-gram negativo** (par `[palabra_seleccionada, palabra_no_cercana]`), para elegir qué palabras NO están asociadas entre si.

Para cada palabra, se generan `num_ns` skip-grams negativos.

La palabra no cercana se selecciona aleatoriamente del vocabulario, puede pasar que se genere un skip-gram-negativo que a la vez exista como skip-gram-positivo.

Resultaría interesante guardar los skip-gram positivos en un set y, al generar los negativos, verificar que no estén en el set.

In [ ]:
# Get target and context words for one positive skip-gram.
target_word, context_word = positive_skip_grams[0]

# Set the number of negative samples per positive context.
num_ns = 4

context_class = tf.reshape(tf.constant(context_word, dtype="int64"), (1, 1))
negative_sampling_candidates, _, _ = tf.random.log_uniform_candidate_sampler(
    true_classes=context_class,  # class that should be sampled as 'positive'
    num_true=1,  # each positive skip-gram has 1 positive context class
    num_sampled=num_ns,  # number of negative context words to sample
    unique=True,  # all the negative samples should be unique
    range_max=vocab_size,  # pick index of the samples from [0, vocab_size]
    seed=SEED,  # seed for reproducibility
    name="negative_sampling"  # name of this operation
)
print(negative_sampling_candidates)
print(inverse_vocab[context_word])
print([inverse_vocab[index.numpy()] for index in negative_sampling_candidates])

tf.Tensor([2 1 4 3], shape=(4,), dtype=int64)
hot
['wide', 'the', 'shimmered', 'road']


### Ejemplo de construcción del train

Dado un **skip-gram positivo** `(target_word, context_word)`, ahora disponemos de `num_ns` **skip-grams negativos**. Se junta el positivo como `1` y los negativos como `0` en un tensor, para generar un set.

In [ ]:
# Reduce a dimension so you can use concatenation (in the next step).
squeezed_context_class = tf.squeeze(context_class, 1)

# Concatenate a positive context word with negative sampled words.
context = tf.concat([squeezed_context_class, negative_sampling_candidates], 0)

# Label the first context word as `1` (positive) followed by `num_ns` `0`s (negative).
label = tf.constant([1] + [0]*num_ns, dtype="int64")
target = target_word

Ejemplo de lo generado arriba:

In [ ]:
print(f"target_index    : {target}")
print(f"target_word     : {inverse_vocab[target_word]}")
print(f"context_indices : {context}")
print(f"context_words   : {[inverse_vocab[c.numpy()] for c in context]}")
print(f"label           : {label}")

target_index    : 5
target_word     : in
context_indices : [6 2 1 4 3]
context_words   : ['hot', 'wide', 'the', 'shimmered', 'road']
label           : [1 0 0 0 0]


Una tupla `(target, context, label)` constituye una instancia del train. `target` representa la palabra a estudiar, `context` las palabras relacionadas y no relacionadas a esta, y `label` representa si las palabras generan un **skip-gram positivo** con un `1` o un **skip-gram negativo** con un `0`.

In [ ]:
print("target  :", target)
print("context :", context)
print("label   :", label)

target  : 5
context : tf.Tensor([6 2 1 4 3], shape=(5,), dtype=int64)
label   : tf.Tensor([1 0 0 0 0], shape=(5,), dtype=int64)


### Resumen de lo anterior

![word2vec_negative_sampling](https://tensorflow.org/text/tutorials/images/word2vec_negative_sampling.png)

### Probabilidades de muestreo

Las stop-words pueden representar un gran problema en word2vec. Una manera de lidiar con ellas es ordenarlas por su idf en una lista, y dependiendo de su posición en la lista, aplicar una probabilidad mayor o menor de añadir un skip-gram en el que estén contenidas.

Una palabra más frecuente tendrá menos probabilidades de ser añadida. Sigue una distribución zipfiana.

In [ ]:
sampling_table = tf.keras.preprocessing.sequence.make_sampling_table(size=10)
print(sampling_table)

[0.00315225 0.00315225 0.00547597 0.00741556 0.00912817 0.01068435
 0.01212381 0.01347162 0.01474487 0.0159558 ]


## Función que realiza todo lo anterior


In [2]:
# Generates skip-gram pairs with negative sampling for a list of sequences
# (int-encoded sentences) based on window size, number of negative samples
# and vocabulary size.
def generate_training_data(sequences, window_size, num_ns, vocab_size, seed):
  # Elements of each training example are appended to these lists.
  targets, contexts, labels = [], [], []

  # Build the sampling table for `vocab_size` tokens.
  sampling_table = tf.keras.preprocessing.sequence.make_sampling_table(vocab_size)

  # Iterate over all sequences (sentences) in the dataset.
  for sequence in tqdm.tqdm(sequences):

    # Generate positive skip-gram pairs for a sequence (sentence).
    positive_skip_grams, _ = tf.keras.preprocessing.sequence.skipgrams(
          sequence,
          vocabulary_size=vocab_size,
          sampling_table=sampling_table,
          window_size=window_size,
          negative_samples=0,
          seed=seed)

    # Iterate over each positive skip-gram pair to produce training examples
    # with a positive context word and negative samples.
    for target_word, context_word in positive_skip_grams:
      context_class = tf.expand_dims(
          tf.constant([context_word], dtype="int64"), 1)
      negative_sampling_candidates, _, _ = tf.random.log_uniform_candidate_sampler(
          true_classes=context_class,
          num_true=1,
          num_sampled=num_ns,
          unique=True,
          range_max=vocab_size,
          seed=seed,
          name="negative_sampling")

      # Build context and label vectors (for one target word)
      context = tf.concat([tf.squeeze(context_class,1), negative_sampling_candidates], 0)
      label = tf.constant([1] + [0]*num_ns, dtype="int64")

      # Append each element from the training example to global lists.
      targets.append(target_word)
      contexts.append(context)
      labels.append(label)

  return targets, contexts, labels

## Preparar los datos para el train

### Importar el texto


Se construye un `tf.data.TextLineDataset` con cada línea no vacía:

In [8]:
text_ds = tf.data.TextLineDataset(path_to_file).filter(lambda x: tf.cast(tf.strings.length(x), bool))

### Vectorizar las frases

Se utiliza `TextVectorization` para vectorizar las frases. Para eliminar signos de puntuación y convertir todo a minúsculas, se usa la función `custom_standardization function` definida abajo.

In [9]:
# Now, create a custom standardization function to lowercase the text and
# remove punctuation.
def custom_standardization(input_data):
  lowercase = tf.strings.lower(input_data)
  return tf.strings.regex_replace(lowercase,
                                  '[%s]' % re.escape(string.punctuation), '')


# Define the vocabulary size and the number of words in a sequence.
vocab_size = 4096
sequence_length = 10

# Use the `TextVectorization` layer to normalize, split, and map strings to
# integers. Set the `output_sequence_length` length to pad all samples to the
# same length.
vectorize_layer = layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length)

Creación del vocabulario y vectores


In [11]:
vectorize_layer.adapt(text_ds.batch(1024))

In [12]:
# Save the created vocabulary for reference.
inverse_vocab = vectorize_layer.get_vocabulary()
print(inverse_vocab[:20])

['', '[UNK]', np.str_('the'), np.str_('was'), np.str_('and'), np.str_('to'), np.str_('he'), np.str_('she'), np.str_('in'), np.str_('had'), np.str_('her'), np.str_('hospital'), np.str_('of'), np.str_('a'), np.str_('that'), np.str_('his'), np.str_('for'), np.str_('we'), np.str_('they'), np.str_('but')]


In [13]:
# Vectorize the data in text_ds.
text_vector_ds = text_ds.batch(1024).prefetch(AUTOTUNE).map(vectorize_layer).unbatch()

### Transformación del dataset

Se convierte el `tf.data.Dataset` a un formato con el que poder generar instancias para el train.

In [14]:
sequences = list(text_vector_ds.as_numpy_iterator())
print(len(sequences))

2234


### Generar instancias para el train

`sequences` ahora contiene frases codificadas. Llamando a la función `generate_training_data` definida anteriormente, se pueden generar instancias para poder llevar a cabo el train.

In [15]:
targets, contexts, labels = generate_training_data(
    sequences=sequences,
    window_size=2,
    num_ns=4,
    vocab_size=vocab_size,
    seed=SEED)

targets = np.array(targets)
contexts = np.array(contexts)
labels = np.array(labels)

print('\n')
print(f"targets.shape: {targets.shape}")
print(f"contexts.shape: {contexts.shape}")
print(f"labels.shape: {labels.shape}")


100%|██████████| 2234/2234 [00:03<00:00, 718.77it/s]




targets.shape: (10464,)
contexts.shape: (10464, 5)
labels.shape: (10464, 5)


### Optimización

Se utiliza un objeto `tf.data.Dataset` que guarda la tubla `(target_word, context_word), (label)` para entrenar el modelo word2vec.

In [16]:
BATCH_SIZE = 1024
BUFFER_SIZE = 10000
dataset = tf.data.Dataset.from_tensor_slices(((targets, contexts), labels))
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)
print(dataset)

<_BatchDataset element_spec=((TensorSpec(shape=(1024,), dtype=tf.int64, name=None), TensorSpec(shape=(1024, 5), dtype=tf.int64, name=None)), TensorSpec(shape=(1024, 5), dtype=tf.int64, name=None))>


Se aplica `Dataset.cache` y `Dataset.prefetch` para optimizar:

In [17]:
dataset = dataset.cache().prefetch(buffer_size=AUTOTUNE)
print(dataset)

<_PrefetchDataset element_spec=((TensorSpec(shape=(1024,), dtype=tf.int64, name=None), TensorSpec(shape=(1024, 5), dtype=tf.int64, name=None)), TensorSpec(shape=(1024, 5), dtype=tf.int64, name=None))>


## Modelo y train

### Subclase de Word2Vec

In [18]:
class Word2Vec(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim):
    super(Word2Vec, self).__init__()
    self.target_embedding = layers.Embedding(vocab_size,
                                      embedding_dim,
                                      name="w2v_embedding")
    self.context_embedding = layers.Embedding(vocab_size,
                                       embedding_dim)

  def call(self, pair):
    target, context = pair
    # target: (batch, dummy?)  # The dummy axis doesn't exist in TF2.7+
    # context: (batch, context)
    if len(target.shape) == 2:
      target = tf.squeeze(target, axis=1)
    # target: (batch,)
    word_emb = self.target_embedding(target)
    # word_emb: (batch, embed)
    context_emb = self.context_embedding(context)
    # context_emb: (batch, context, embed)
    dots = tf.einsum('be,bce->bc', word_emb, context_emb)
    # dots: (batch, context)
    return dots

### Compilar el modelo


In [19]:
embedding_dim = DIM_VECTORES
word2vec = Word2Vec(vocab_size, embedding_dim)
word2vec.compile(optimizer='adam',
                 loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                 metrics=['accuracy'])

Train the model on the `dataset` for some number of epochs:

In [20]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="logs")
word2vec.fit(dataset, epochs=20, callbacks=[tensorboard_callback])

Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.2343 - loss: 1.6085
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5481 - loss: 1.5981
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7104 - loss: 1.5860
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7779 - loss: 1.5684
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7995 - loss: 1.5420
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7956 - loss: 1.5051
Epoch 7/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7848 - loss: 1.4588
Epoch 8/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7722 - loss: 1.4060
Epoch 9/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7657 - loss: 1.3493
Epoch 10/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7642 - loss: 1.2907
Epoch 11/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7648 - loss: 1.2315
Epoch 12/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy:

## Análisis

Se obtienen las palabras del vocabulario y sus respectivos vectores

In [21]:
weights = word2vec.get_layer('w2v_embedding').get_weights()[0]
vocab = vectorize_layer.get_vocabulary()

Generar y guardar los archivos con los vectores y metadatos:

In [22]:
out_v = io.open('vectors.tsv', 'w', encoding='utf-8')
out_m = io.open('metadata.tsv', 'w', encoding='utf-8')

for index, word in enumerate(vocab):
  if index == 0:
    continue  # skip 0, it's padding.
  vec = weights[index]
  out_v.write('\t'.join([str(x) for x in vec]) + "\n")
  out_m.write(word + "\n")
out_v.close()
out_m.close()

In [23]:
try:
  from google.colab import files
  files.download('vectors.tsv')
  files.download('metadata.tsv')
except Exception:
  pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>